In [3]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# ── Load environment variables ────────────────────────────────────────────────
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
print("OpenAI key loaded:", "✅" if api_key else "❌ MISSING")

# ── Paths ─────────────────────────────────────────────────────────────────────
ROOT       = Path.cwd().parent
PDF_DIR    = ROOT / 'data' / 'pdfs'
VECTOR_DIR = ROOT / 'vectorstore'

print(f'PDFs folder  : {PDF_DIR}')
print(f'Vector store : {VECTOR_DIR}')

OpenAI key loaded: ✅
PDFs folder  : D:\rag-chatbot\data\pdfs
Vector store : D:\rag-chatbot\vectorstore


In [4]:
# ── List available PDFs ───────────────────────────────────────────────────────
pdfs = list(PDF_DIR.glob('*.pdf'))
print(f'Found {len(pdfs)} PDF(s):')
for p in pdfs:
    size_mb = p.stat().st_size / 1024 / 1024
    print(f'  {p.name} ({size_mb:.1f} MB)')

if len(pdfs) == 0:
    print("❌ No PDFs found — check data/pdfs/ folder")

Found 7 PDF(s):
  dib-lens-quarterly-report-july-2025.pdf (2.8 MB)
  Dubai-Residential-Market-Performance-Q12025.pdf (7.0 MB)
  dubai-residential-market-report---q2-2025.pdf (2.1 MB)
  dubai-residential-market-review-q1-2026.pdf (4.3 MB)
  dubai-residential-market-review-special-edition-q3-02025.pdf (19.1 MB)
  dubai-residential-market-review-spring-summer-2024-11268.pdf (8.9 MB)
  july_market_report_2025.pdf (18.9 MB)


In [5]:
from langchain_community.document_loaders import PyPDFLoader

# ── Load all PDFs into LangChain Document objects ─────────────────────────────
# Each page becomes one Document:
#   page_content: text on that page
#   metadata: source filename + page number
all_docs = []

for pdf_path in pdfs:
    print(f'Loading {pdf_path.name}...')
    try:
        loader = PyPDFLoader(str(pdf_path))
        docs   = loader.load()
        all_docs.extend(docs)
        print(f'  → {len(docs)} pages loaded')
    except Exception as e:
        print(f'  ❌ Failed: {e}')

print(f'\nTotal pages loaded: {len(all_docs)}')

Loading dib-lens-quarterly-report-july-2025.pdf...


C:\Users\ahmed\anaconda3\envs\tensorflow\lib\site-packages\pypdf\_crypt_providers\_cryptography.py:32: CryptographyDeprecationWarning: ARC4 has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.ARC4 and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  from cryptography.hazmat.primitives.ciphers.algorithms import AES, ARC4


  → 10 pages loaded
Loading Dubai-Residential-Market-Performance-Q12025.pdf...
  → 21 pages loaded
Loading dubai-residential-market-report---q2-2025.pdf...
  → 5 pages loaded
Loading dubai-residential-market-review-q1-2026.pdf...
  → 5 pages loaded
Loading dubai-residential-market-review-special-edition-q3-02025.pdf...
  → 22 pages loaded
Loading dubai-residential-market-review-spring-summer-2024-11268.pdf...
  → 3 pages loaded
Loading july_market_report_2025.pdf...
  → 34 pages loaded

Total pages loaded: 100


In [6]:
# ── Check what a loaded document looks like ───────────────────────────────────
sample = all_docs[2]
print('── Metadata ──────────────────────────────────────')
print(sample.metadata)
print('\n── Content preview (first 500 chars) ────────────')
print(sample.page_content[:500])

── Metadata ──────────────────────────────────────
{'source': 'D:\\rag-chatbot\\data\\pdfs\\dib-lens-quarterly-report-july-2025.pdf', 'page': 2}

── Content preview (first 500 chars) ────────────
Dubai’s residential real estate market continued its 
record-breaking run in H1-2025, with transaction volumes and 
values hitting new highs despite global economic headwinds 
and regional geopolitical uncertainty. In Q2-2025, Dubai 
recorded an all-time high of 52,103 residential real estate 
transactions with total transactions value reaching   156.7 
billion - reﬂecting robust demand & heightened investor 
conﬁdence in the sector.
The UAE economy is forecasted to accelerate to 4.6% GDP 
growt


In [7]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# ── Why chunking? ─────────────────────────────────────────────────────────────
# LLMs have a context window limit.
# We split into overlapping chunks so:
#   - Each chunk fits in the LLM context
#   - 200 char overlap prevents losing info at chunk boundaries

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,       # ~250 tokens per chunk
    chunk_overlap=200,     # overlap between consecutive chunks
    separators=['\n\n', '\n', '.', ' ', ''],
)

chunks = splitter.split_documents(all_docs)

print(f'Total pages  : {len(all_docs)}')
print(f'Total chunks : {len(chunks)}')
print(f'Avg chunk    : {sum(len(c.page_content) for c in chunks) // len(chunks)} chars')

Total pages  : 100
Total chunks : 271
Avg chunk    : 770 chars


In [8]:
# ── See what one chunk looks like ─────────────────────────────────────────────
sample_chunk = chunks[10]
print('── Metadata ──────────────────────────────────────')
print(sample_chunk.metadata)
print('\n── Content ───────────────────────────────────────')
print(sample_chunk.page_content)
print(f'\nLength: {len(sample_chunk.page_content)} chars')

── Metadata ──────────────────────────────────────
{'source': 'D:\\rag-chatbot\\data\\pdfs\\dib-lens-quarterly-report-july-2025.pdf', 'page': 5}

── Content ───────────────────────────────────────
6QUARTERLY REPORT - JULY 2025254 new projects launched in H1-2025 reﬂecting strong investor
demand & developer conﬁdenceUpcoming Project Launches
(H1-2025)
Pricing Trends:
Steady Growth in Average Price Per
Sq. M. as demand continues to
outpace supply
Source: Dubai Land Department, Dubai Pulse Source: Dubai Land DepartmentSource: Dubai Land Department, Dubai Pulse
Source: Dubai Land Department, Dubai PulseNew Projects Launched (1H 2025)Highest-Valued Projects Launched
Units Under Construction
Most Active Zone Authority for Real Estate
Project DevelopmentAverage Price Per Square Meter (   )
peaked in Q1 2024
Dubai Development Authority
Trakhees
Dubai Municipality
Dubai South254 ProjectsTotalQ1-2025 Q2-2025
Eden Hills
Q2-2025  19,660.8                 +2.8%    +6.1%
Q1-2025  19,118.5           

In [9]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
import shutil

# ── OpenAI embeddings ─────────────────────────────────────────────────────────
# text-embedding-3-small: best price/performance for retrieval.
# Converts each chunk to a 1536-dimensional vector.
# Cost: ~$0.001 per 1M tokens — your PDFs cost less than $0.01 total.
embeddings = OpenAIEmbeddings(
    model='text-embedding-3-small',
    openai_api_key=api_key
)

# ── Clear existing vectorstore if rebuilding ──────────────────────────────────
if VECTOR_DIR.exists():
    shutil.rmtree(VECTOR_DIR)
    print('Cleared existing vector store')

# ── Embed all chunks and store in ChromaDB ────────────────────────────────────
# Sends chunks to OpenAI API, gets vectors back, stores on disk.
print(f'Embedding {len(chunks)} chunks...')
print('This takes 1-2 minutes...')

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=str(VECTOR_DIR),
)

print(f'\n✅ Vector store built → {VECTOR_DIR}')
print(f'Total vectors: {vectorstore._collection.count()}')

Cleared existing vector store
Embedding 271 chunks...
This takes 1-2 minutes...


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given



✅ Vector store built → D:\rag-chatbot\vectorstore
Total vectors: 271


In [10]:
# ── Verify semantic search works ──────────────────────────────────────────────
query   = "What is the average property price in Dubai?"
results = vectorstore.similarity_search(query, k=3)

print(f'Query: "{query}"\n')
for i, doc in enumerate(results):
    print(f'── Result {i+1} ──────────────────────────────────')
    print(f'Source : {Path(doc.metadata.get("source", "?")).name}')
    print(f'Page   : {doc.metadata.get("page", "?")}')
    print(f'Preview: {doc.page_content[:200]}...')
    print()

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Query: "What is the average property price in Dubai?"

── Result 1 ──────────────────────────────────
Source : july_market_report_2025.pdf
Page   : 23
Preview: In terms of rent levels, the trend in July 
continued to be one of moderation compared to the steep increases of 2022–2023. Market reports and portal data show that rent rises have cooled to single-di...

── Result 2 ──────────────────────────────────
Source : dubai-residential-market-review-special-edition-q3-02025.pdf
Page   : 12
Preview: and currently average AED 65,000 p.a. for new contracts, 
leaving them 13.1% above the 2014 peak. A typical two-bedroom apartment now commands AED 70,000–75,000 
p.a. (c. AED 700-900 psf p.a.), while ...

── Result 3 ──────────────────────────────────
Source : dubai-residential-market-review-q1-2026.pdf
Page   : 3
Preview: Dubai Residential Market Review - Q1 2026 Dubai Residential Market Review - Q1 20266 7RESIDENTIAL VALUES IN DUBAI
Average AED psf
Source: Knight Frank, REIDINAl Barari
Q1 2

In [11]:
test_queries = [
    "What are RERA regulations for property buyers?",
    "Which areas have the highest property prices?",
    "What is the outlook for Dubai real estate?",
]

for query in test_queries:
    results = vectorstore.similarity_search(query, k=1)
    print(f'Q: {query}')
    if results:
        print(f'→ Source : {Path(results[0].metadata.get("source","?")).name}')
        print(f'→ Preview: {results[0].page_content[:150]}...')
    print()

Q: What are RERA regulations for property buyers?
→ Source : july_market_report_2025.pdf
→ Preview: The new First-Time Home Buyer Programme is 
expected to inject a fresh wave of demand at the lower end, as previously discussed – eﬀectively turning m...

Q: Which areas have the highest property prices?
→ Source : dubai-residential-market-review-special-edition-q3-02025.pdf
→ Preview: cohort of high-income residents seeking larger family-
oriented spaces.
On an annual basis however, luxury communities were once 
again at the forefro...

Q: What is the outlook for Dubai real estate?
→ Source : july_market_report_2025.pdf
→ Preview: In summary, the medium-term outlook for Dubai 
real estate is one of tempered optimism. The market is transitioning from an extraordinary growth spurt...

